# Plot the trained model's predicted surfaces with the actual dataset surfaces

In [1]:
import ase.io
import os
from pathlib import Path
import numpy as np
import importlib
import torch
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist
from scipy.linalg import orthogonal_procrustes

import sampling_methods.descriptors as descriptors
import sampling_methods.selectors as selectors

importlib.reload(descriptors)
importlib.reload(selectors)


<module 'sampling_methods.selectors' from '/home/lim_yt/X-MACE-sampling/sampling_methods/selectors.py'>

In [2]:
XYZ = "../data/A02_propene/static/A02_propene_grid_static_CASSCF_adjusted_2.xyz"
N_GEOMETRIES = 3731  # number of geometries to use from the dataset
atoms_list = ase.io.read(XYZ, index=f":{N_GEOMETRIES}")
print("atoms_list length:", len(atoms_list))

bond_lengths = []
dihedrals = []
energy_surfaces = None

bond_lengths = np.asarray([descriptors.get_descriptor("bond_lengths", atoms)[0] for atoms in atoms_list])
dihedrals = np.asarray([descriptors.get_descriptor("dihedral", atoms)[0] for atoms in atoms_list])
energies = np.vstack([np.asarray(atoms.info["REF_energy"]).ravel() for atoms in atoms_list])
energy_surfaces = [energies[:, state] for state in range(energies.shape[1])]
print("Number of energy surfaces:", len(energy_surfaces))

atoms_list length: 3731
Number of energy surfaces: 3


In [3]:
# Path to the trained model to evaluate.
MODEL = "../outputs/base_models/base_model_run_33_fold_3.pt"
INFERENCE_BATCH_SIZE = 64
R_MAX = 5.0

# Make project modules available when this notebook runs from notebooks/.
import sys
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.helper import _import_project_modules, _load_model

model_path = Path(MODEL).resolve()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_, AtomDataLoaderBuilder, *_ = _import_project_modules()

data_loader = AtomDataLoaderBuilder(
    cutoff=R_MAX, energy_key="REF_energy", forces_key="REF_forces"
).load(atoms_list, batch_size=INFERENCE_BATCH_SIZE, shuffle=False)

model = _load_model(model_path, device)
model.eval()

predicted_batches = []
for batch in data_loader:
    batch = batch.to(device)
    # Do not use torch.no_grad(): this model computes gradients internally,
    # even when compute_force is False.
    output = model(batch.to_dict(), training=False, compute_force=False)
    predicted_batches.append(output["energy"].detach().cpu())

predicted_energies = torch.cat(predicted_batches).numpy()
if predicted_energies.shape != energies.shape:
    raise ValueError(
        "Model predictions must have shape "
        f"{energies.shape} (geometries, states), but got "
        f"{predicted_energies.shape}."
    )

predicted_energy_surfaces = [
    predicted_energies[:, state] for state in range(predicted_energies.shape[1])
]
print("Predicted energy shape:", predicted_energies.shape)


/home/lim_yt/micromamba/envs/xmace311/lib/python3.11/site-packages/h5py/__init__.py:36: UserWarning: h5py is running against HDF5 2.2.0 when it was built against 2.1.0, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "


KeyboardInterrupt: 

In [ ]:
# Plot the actual (green) and predicted (red) surfaces for each state.
%matplotlib inline

fig, axes = plt.subplots(
    1, len(energy_surfaces), figsize=(7 * len(energy_surfaces), 6),
    subplot_kw={"projection": "3d"}, constrained_layout=True, squeeze=False,
)
for state, ax in enumerate(axes.ravel()):
    ax.scatter(
        bond_lengths, dihedrals, energy_surfaces[state],
        color="tab:green", s=6, alpha=0.55, label="Actual",
    )
    ax.scatter(
        bond_lengths, dihedrals, predicted_energy_surfaces[state],
        color="tab:red", s=6, alpha=0.55, label="Predicted",
    )
    ax.set_xlabel("Bond length (Å)")
    ax.set_ylabel("Dihedral angle (degrees)")
    ax.set_zlabel("Energy")
    ax.set_title(f"S{state}")
    ax.legend()

fig.suptitle("Predicted (red) and actual (green) energy surfaces", y=1.02)
plt.show()


In [ ]:
# Plot prediction residuals: predicted energy minus actual energy.
%matplotlib inline

energy_residual_surfaces = [
    np.abs(predicted_surface - actual_surface)
    for predicted_surface, actual_surface in zip(
        predicted_energy_surfaces, energy_surfaces
    )
]
residual_limit = max(
    np.max(np.abs(residual_surface))
    for residual_surface in energy_residual_surfaces
)
print("Highest residual:", residual_limit)

fig, axes = plt.subplots(
    1, len(energy_residual_surfaces), figsize=(7 * len(energy_residual_surfaces), 6),
    subplot_kw={"projection": "3d"}, constrained_layout=True, squeeze=False,
)
for state, ax in enumerate(axes.ravel()):
    residual_plot = ax.scatter(
        bond_lengths, dihedrals, energy_residual_surfaces[state],
        c=energy_residual_surfaces[state], cmap="coolwarm",
        vmin=0, vmax=residual_limit, s=8, alpha=0.7,
    )
    fig.colorbar(residual_plot, ax=ax, label="Predicted − actual energy")
    ax.set_xlabel("Bond length (Å)")
    ax.set_ylabel("Dihedral angle (degrees)")
    ax.set_zlabel("Predicted − actual energy")
    ax.set_title(f"S{state} residual")

fig.suptitle("Energy-surface residuals: predicted − actual", y=1.02)
plt.show()

In [ ]:
# Plot prediction residuals: predicted energy minus actual energy.
%matplotlib inline

energy_residual_surfaces = [
    np.abs(predicted_surface - actual_surface)
    for predicted_surface, actual_surface in zip(
        predicted_energy_surfaces, energy_surfaces
    )
]
residual_limit = max(
    np.max(np.abs(residual_surface))
    for residual_surface in energy_residual_surfaces
)
print("Highest residual:", residual_limit)

bond_grid = np.unique(np.round(bond_lengths, 8))
dihedral_grid = np.unique(np.round(dihedrals, 8))
bond_idx = np.searchsorted(bond_grid, np.round(bond_lengths, 8))
dihedral_idx = np.searchsorted(dihedral_grid, np.round(dihedrals, 8))

fig, axes = plt.subplots(
    1, len(energy_residual_surfaces), figsize=(6 * len(energy_residual_surfaces), 5),
    sharex=True, sharey=True, constrained_layout=True, squeeze=False,
)
for state, ax in enumerate(axes.ravel()):
    residual_grid = np.full((len(dihedral_grid), len(bond_grid)), np.nan)
    residual_grid[dihedral_idx, bond_idx] = energy_residual_surfaces[state]
    residual_plot = ax.pcolormesh(
        bond_grid, dihedral_grid, residual_grid, shading="nearest",
        cmap="coolwarm", vmin=0, vmax=residual_limit,
    )
    fig.colorbar(residual_plot, ax=ax, label="Predicted − actual energy")
    ax.set_xlabel("Bond length (Å)")
    ax.set_title(f"S{state} residual")

axes[0, 0].set_ylabel("Dihedral angle (degrees)")
fig.suptitle("Energy-surface residuals: predicted − actual", y=1.02)
plt.show()


## Base-model fold prediction variance

Evaluate the five base-model folds from `run_1` on their A03-butene training grid and plot their prediction variance.

In [1]:
%matplotlib inline

import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import ase.io
import matplotlib.pyplot as plt
import numpy as np
import torch

import sampling_methods.descriptors as descriptors

BUTENE_GRID = Path("../data/A01_ethene/static/A01_ethene_grid_static_CASSCF_FINAL.xyz")
RUN_DIRECTORY = Path("../scripts/output/run_0")
N_GEOMETRIES = 3731
INFERENCE_BATCH_SIZE = 64
R_MAX = 5.0

butene_atoms = ase.io.read(BUTENE_GRID, index=f":{N_GEOMETRIES}")
fold_model_paths = [
    RUN_DIRECTORY / f"base_model_fold_{fold}.pt"
    for fold in range(1, 3)
]


In [2]:

from scripts.helper import _import_project_modules, _load_model

_, AtomDataLoaderBuilder, *_ = _import_project_modules()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
butene_loader = AtomDataLoaderBuilder(
    cutoff=R_MAX, energy_key="REF_energy", forces_key="REF_forces"
).load(butene_atoms, batch_size=INFERENCE_BATCH_SIZE, shuffle=False)

fold_predictions = []
for fold, model_path in enumerate(fold_model_paths, start=1):
    model = _load_model(model_path.resolve(), device)
    model.eval()

    prediction_batches = []
    for batch in butene_loader:
        batch = batch.to(device)
        output = model(batch.to_dict(), training=False, compute_force=False)
        prediction_batches.append(output["energy"].detach().cpu())

    fold_predictions.append(torch.cat(prediction_batches).numpy())
    print(f"Completed predictions for base-model fold {fold}")


/home/lim_yt/micromamba/envs/xmace311/lib/python3.11/site-packages/h5py/__init__.py:36: UserWarning: h5py is running against HDF5 2.2.0 when it was built against 2.1.0, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "


NotImplementedError: Could not run 'aten::empty_strided' with arguments from the 'CUDA' backend. This could be because the operator doesn't exist for this backend, or was omitted during the selective/custom build process (if using custom build). If you are a Facebook employee using PyTorch on mobile, please visit https://fburl.com/ptmfixes for possible resolutions. 'aten::empty_strided' is only available for these backends: [CPU, Meta, QuantizedCPU, BackendSelect, Python, FuncTorchDynamicLayerBackMode, Functionalize, Named, Conjugate, Negative, ZeroTensor, ADInplaceOrView, AutogradOther, AutogradCPU, AutogradCUDA, AutogradHIP, AutogradXLA, AutogradMPS, AutogradIPU, AutogradXPU, AutogradHPU, AutogradVE, AutogradLazy, AutogradMTIA, AutogradMAIA, AutogradPrivateUse1, AutogradPrivateUse2, AutogradPrivateUse3, AutogradMeta, AutogradNestedTensor, Tracer, AutocastCPU, AutocastMTIA, AutocastMAIA, AutocastXPU, AutocastMPS, AutocastCUDA, FuncTorchBatched, BatchedNestedTensor, FuncTorchVmapMode, 

In [ ]:

ensemble_predictions = np.stack(fold_predictions, axis=0)
prediction_variance = np.var(ensemble_predictions, axis=0, ddof=1)

butene_bond_lengths = np.asarray([
    descriptors.get_descriptor("bond_lengths", atoms)[0]
    for atoms in butene_atoms
])
butene_dihedrals = np.asarray([
    descriptors.get_descriptor("dihedral", atoms)[0]
    for atoms in butene_atoms
])
bond_grid = np.unique(np.round(butene_bond_lengths, 8))
dihedral_grid = np.unique(np.round(butene_dihedrals, 8))
bond_idx = np.searchsorted(bond_grid, np.round(butene_bond_lengths, 8))
dihedral_idx = np.searchsorted(dihedral_grid, np.round(butene_dihedrals, 8))

variance_grids = np.full(
    (prediction_variance.shape[1], len(dihedral_grid), len(bond_grid)), np.nan
)
variance_grids[:, dihedral_idx, bond_idx] = prediction_variance.T
variance_limit = np.nanmax(variance_grids)

fig, axes = plt.subplots(
    1, variance_grids.shape[0],
    figsize=(6 * variance_grids.shape[0], 5),
    sharex=True, sharey=True, constrained_layout=True, squeeze=False,
)
for state, ax in enumerate(axes.ravel()):
    variance_plot = ax.pcolormesh(
        bond_grid, dihedral_grid, variance_grids[state],
        shading="nearest", cmap="magma", vmin=0, vmax=variance_limit,
    )
    ax.set_xlabel("Bond length (Å)")
    ax.set_title(f"S{state} prediction variance")

axes[0, 0].set_ylabel("Dihedral angle (degrees)")
fig.colorbar(variance_plot, ax=axes.ravel(), label="Energy prediction variance (eV²)")
fig.suptitle("Base-model fold prediction variance", y=1.02)
plt.show()
